In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!pip -q install duckdb pandas transformers accelerate bitsandbytes


In [ ]:
DB_PATH = "/content/drive/MyDrive/CricketLLM/ipl_full.duckdb"
PROMPT_PATH = "/content/drive/MyDrive/CricketLLM/sql_agent_master-prompt_compressed_v2.txt."
FUNC_PATH = "/content/drive/MyDrive/CricketLLM/function.py"


In [ ]:
import importlib.util
from pathlib import Path

spec = importlib.util.spec_from_file_location("function", FUNC_PATH)
function = importlib.util.module_from_spec(spec)
spec.loader.exec_module(function)

In [ ]:
import duckdb
con = duckdb.connect(DB_PATH, read_only=True)

print(con.execute("SHOW TABLES").fetchdf())
print(con.execute("SELECT * FROM player_batting_stats LIMIT 3").fetchdf())

con.close()


                       name
0    batter_bowler_matchups
1                deliveries
2             match_summary
3                   matches
4      player_batting_stats
5      player_bowling_stats
6  team_phase_batting_stats
7  team_phase_bowling_stats
          batter  innings  dismissals    runs  balls_faced  fours  sixes  \
0       CH Gayle      145       125.0  4997.0       3331.0  408.0  359.0   
1  Mandeep Singh       97        80.0  1706.0       1385.0  176.0   38.0   
2        TM Head       37        32.0  1146.0        667.0  126.0   55.0   

   dot_balls  strike_rate  batting_average  dot_ball_pct  boundary_ball_pct  
0     1462.0   150.015011          39.9760     43.890724          23.026118  
1      523.0   123.176895          21.3250     37.761733          15.451264  
2      218.0   171.814093          35.8125     32.683658          27.136432  


In [ ]:
import re

def load_master_prompt(path: str) -> str:
    with open(path, "r", encoding="utf-8") as f:
        return f.read()


def clean_sql(text: str) -> str:
    t = text.strip()

    # Remove markdown fences
    t = re.sub(r"```sql", "", t, flags=re.IGNORECASE).strip()
    t = re.sub(r"```", "", t).strip()

    # Find last SQL start keyword (SELECT or WITH)
    m = list(re.finditer(r"\b(SELECT|WITH)\b", t, flags=re.IGNORECASE))
    if not m:
        return t.strip()

    start = m[-1].start()
    sql = t[start:].strip()

    # If it contains multiple statements, keep first statement
    # Prefer ending at semicolon if present
    semi = sql.find(";")
    if semi != -1:
        sql = sql[:semi+1].strip()

    return sql



In [ ]:
import pandas as pd

def run_sql_in_duckdb(sql: str, db_path: str = DB_PATH) -> pd.DataFrame:
    con = duckdb.connect(db_path, read_only=True)
    try:
        return con.execute(sql).fetchdf()
    finally:
        con.close()


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-Coder-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    load_in_4bit=True,
    torch_dtype=torch.float16
)

print("Model loaded. CUDA available:", torch.cuda.is_available())


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00004-of-00004.safetensors:   0%|          | 0.00/1.09G [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.88G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded. CUDA available: True


In [ ]:
def call_hf_for_sql(question: str, master_prompt: str) -> str:
    messages = [
        {"role": "system", "content": master_prompt},
        {"role": "user", "content": question},
    ]

    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.0,      # deterministic
            do_sample=False,      # deterministic
            pad_token_id=tokenizer.eos_token_id
        )

    # ✅ ONLY decode the generated part (not the prompt)
    gen_ids = output_ids[0][input_len:]
    decoded = tokenizer.decode(gen_ids, skip_special_tokens=True)

    return decoded.strip()


In [ ]:
def ask_cricket_question(question: str, verbose: bool = True):
    master_prompt = load_master_prompt(PROMPT_PATH)

    raw_output = call_hf_for_sql(question, master_prompt)
    sql = clean_sql(raw_output)

    if verbose:
        print("=== Raw LLM output ===")
        print(raw_output)
        print("======================")
        print("=== Final SQL to execute ===")
        print(sql)
        print("============================")

    df = run_sql_in_duckdb(sql, DB_PATH)
    return df, sql, raw_output

In [ ]:
df = ask_cricket_question("How many runs has RG Sharma scored?")
df

=== Raw LLM output ===
```sql
SELECT runs AS total_runs
FROM player_batting_stats
WHERE batter = 'RG Sharma';
```
=== Final SQL to execute ===
SELECT runs AS total_runs
FROM player_batting_stats
WHERE batter = 'RG Sharma';


,total_runs
0,7048.0


In [ ]:
def ask_cricket_question_with_retry(question: str, max_retries: int = 1):
    master_prompt = load_master_prompt(PROMPT_PATH)

    # 1) First attempt
    raw1 = call_hf_for_sql(question, master_prompt)
    sql1 = clean_sql(raw1)

    try:
        df1 = run_sql_in_duckdb(sql1, DB_PATH)
        return df1, sql1, ""   # success
    except Exception as e1:
        err1 = str(e1)

    # 2) Retry (only once)
    if max_retries <= 0:
        return None, sql1, err1

    repair_prompt = master_prompt + "\n\n" + (
        "The SQL you generated caused this database error:\n"
        f"{err1}\n\n"
        "Fix the SQL to match the schema/views and output ONLY the corrected SQL."
    )

    raw2 = call_hf_for_sql(question, repair_prompt)
    sql2 = clean_sql(raw2)

    try:
        df2 = run_sql_in_duckdb(sql2, DB_PATH)
        return df2, sql2, ""  # success after retry
    except Exception as e2:
        return None, sql2, str(e2)

Evaluation:

1st step : Exceuction validity (syntax and logic correct)

2nd step : Semantic correctness (correct answer)

In [ ]:
import pandas as pd

EVAL_PATH = "/content/drive/MyDrive/CricketLLM/eval_questions_v1.csv"

eval_rows = [
    {"id": 1, "question": "How many runs has RG Sharma scored?",
     "expected_view": "player_batting_stats", "category": "player_batting"},

    {"id": 2, "question": "What are V Kohli's runs, balls faced and strike rate?",
     "expected_view": "player_batting_stats", "category": "player_batting"},

    {"id": 3, "question": "Give JJ Bumrah's overs, runs conceded, wickets, economy, bowling average, and bowling strike rate in the IPL.",
     "expected_view": "player_bowling_stats", "category": "player_bowling"},

    {"id": 4, "question": "What is JJ Bumrah's economy rate and wickets taken in death overs across all IPL seasons?",
     "expected_view": "deliveries", "category": "phase_player_bowling"},  # because your phase-bowler view doesn't exist

    {"id": 5, "question": "Show V Kohli's stats against JJ Bumrah.",
     "expected_view": "batter_bowler_matchups", "category": "matchup"},

    {"id": 6, "question": "What were Mumbai Indians' total runs in the powerplay in 2019?",
     "expected_view": "team_phase_batting_stats", "category": "team_phase"},

    {"id": 7, "question": "Give me the head-to-head record between Mumbai Indians and Chennai Super Kings.",
     "expected_view": "match_summary", "category": "head_to_head"},
]

df_eval = pd.DataFrame(eval_rows)
df_eval.to_csv(EVAL_PATH, index=False)
df_eval

,id,question,expected_view,category
0,1,How many runs has RG Sharma scored?,player_batting_stats,player_batting
1,2,"What are V Kohli's runs, balls faced and strik...",player_batting_stats,player_batting
2,3,"Give JJ Bumrah's overs, runs conceded, wickets...",player_bowling_stats,player_bowling
3,4,What is JJ Bumrah's economy rate and wickets t...,deliveries,phase_player_bowling
4,5,Show V Kohli's stats against JJ Bumrah.,batter_bowler_matchups,matchup
5,6,What were Mumbai Indians' total runs in the po...,team_phase_batting_stats,team_phase
6,7,Give me the head-to-head record between Mumbai...,match_summary,head_to_head


In [ ]:
import re

def detect_used_sources(sql: str):
    """
    Very simple parser: extract table/view names used after FROM/JOIN.
    Works well enough for our eval.
    """
    s = re.sub(r"\s+", " ", sql.strip(), flags=re.M)
    tokens = re.findall(r"\b(?:FROM|JOIN)\s+([a-zA-Z_][a-zA-Z0-9_]*)\b", s, flags=re.IGNORECASE)
    # normalize
    return list(dict.fromkeys([t.lower() for t in tokens]))  # preserve order, unique

In [ ]:
import time
import pandas as pd

RESULTS_PATH = "/content/drive/MyDrive/CricketLLM/eval_results_v1.csv"

def run_eval(eval_csv: str = EVAL_PATH):
    df_eval = pd.read_csv(eval_csv)
    results = []

    for _, r in df_eval.iterrows():
        qid = int(r["id"])
        question = r["question"]
        expected_view = str(r["expected_view"]).lower()
        category = r.get("category", "")

        t0 = time.time()
        sql = ""
        used_sources = []
        exec_ok = False
        view_ok = False
        err = ""
        row_count = 0
        retries_used = 0

        df, sql_final, err_final = ask_cricket_question_with_retry(question, max_retries=1)
        sql = sql_final
        err = err_final

        used_sources = detect_used_sources(sql)
        view_ok = expected_view in used_sources

        if df is not None and err == "":
            exec_ok = True
            row_count = int(df.shape[0])

        results.append({
            "id": qid,
            "category": category,
            "question": question,
            "expected_view": expected_view,
            "used_sources": ",".join(used_sources),
            "view_ok": view_ok,
            "exec_ok": exec_ok,
            "row_count": row_count,
            "latency_sec": round(time.time() - t0, 3),
            "sql": sql,
            "error": err
        })

    df_res = pd.DataFrame(results)
    df_res.to_csv(RESULTS_PATH, index=False)
    return df_res

In [ ]:
df_results = run_eval()
print("Execution success:", round(df_results["exec_ok"].mean()*100, 1), "%")
print("View accuracy:", round(df_results["view_ok"].mean()*100, 1), "%")
df_results[["id","expected_view","used_sources","view_ok","exec_ok","error"]]

Execution success: 85.7 %
View accuracy: 100.0 %


,id,expected_view,used_sources,view_ok,exec_ok,error
0,1,player_batting_stats,player_batting_stats,True,True,
1,2,player_batting_stats,player_batting_stats,True,True,
2,3,player_bowling_stats,player_bowling_stats,True,False,"Binder Error: Referenced column ""dismissal_bow..."
3,4,deliveries,deliveries,True,True,
4,5,batter_bowler_matchups,batter_bowler_matchups,True,True,
5,6,team_phase_batting_stats,"team_phase_batting_stats,matches",True,True,
6,7,match_summary,match_summary,True,True,


from matplotlib import pyplot as plt
_df_21['id'].plot(kind='hist', bins=20, title='id')
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
import seaborn as sns
_df_22.groupby('expected_view').size().plot(kind='barh', color=sns.palettes.mpl_palette('Dark2'))
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
import seaborn as sns
_df_23.groupby('used_sources').size().plot(kind='barh', color=sns.palettes.mpl_palette('Dark2'))
plt.gca().spines[['top', 'right',]].set_visible(False)

...

from matplotlib import pyplot as plt
import seaborn as sns
figsize = (12, 1.2 * len(_df_55['used_sources'].unique()))
plt.figure(figsize=figsize)
sns.violinplot(_df_55, x='index', y='used_sources', inner='stick', palette='Dark2')
sns.despine(top=True, right=True, bottom=True, left=True)

<string>:5: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.



from matplotlib import pyplot as plt
import seaborn as sns
figsize = (12, 1.2 * len(_df_56['exec_ok'].unique()))
plt.figure(figsize=figsize)
sns.violinplot(_df_56, x='index', y='exec_ok', inner='stick', palette='Dark2')
sns.despine(top=True, right=True, bottom=True, left=True)

<string>:5: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.



from matplotlib import pyplot as plt
import seaborn as sns
figsize = (12, 1.2 * len(_df_57['error'].unique()))
plt.figure(figsize=figsize)
sns.violinplot(_df_57, x='index', y='error', inner='stick', palette='Dark2')
sns.despine(top=True, right=True, bottom=True, left=True)

# Gradio

In [ ]:
!pip install gradio

In [ ]:
#backend function
def ask_and_return(question):
    try:
        df = ask_cricket_question(question)
        return df.to_markdown(index=False)
    except Exception as e:
        return f"Error:\n{str(e)}"

In [ ]:
import gradio as gr

def run_query_ui(question: str, show_sql: bool):
    try:
        df, sql, raw = ask_cricket_question(question, verbose=False)

        # show only first N rows to keep UI fast
        df_show = df.head(200)

        return (
            df_show,                          # dataframe
            (sql if show_sql else ""),        # sql box
            ""                                # error box
        )
    except Exception as e:
        return (
            None,
            "",
            str(e)
        )

with gr.Blocks(title="IPL SQL Assistant", theme=gr.themes.Soft()) as demo:
    gr.Markdown("## 🏏 IPL Analytics Assistant ")
    gr.Markdown("Ask a question. The system generates SQL and runs it on DuckDB. Result appears below.")

    with gr.Row():
        question = gr.Textbox(
            label="Question",
            placeholder="e.g. How many runs has V Kohli scored?",
            scale=4
        )
        run_btn = gr.Button("Run", variant="primary", scale=1)

    show_sql = gr.Checkbox(value=True, label="Show generated SQL")

    result_df = gr.Dataframe(label="Result", wrap=True)
    sql_box = gr.Code(label="Generated SQL", language="sql", interactive=False)
    error_box = gr.Textbox(label="Error", lines=3)

    run_btn.click(
        fn=run_query_ui,
        inputs=[question, show_sql],
        outputs=[result_df, sql_box, error_box]
    )

demo.launch(share=True, debug=False)

/tmp/ipython-input-1823422611.py:22: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="IPL SQL Assistant", theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://622407b7932db04d16.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


NameError: name 'hello' is not defined